In [1]:
import os
import ast
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from pattern_recognition.data.loadTraces import LoadTraces
from pattern_recognition.utils.parameters_dics import dbTableNames
from dash_viewer.classification.load_prepare_data import load_queries_table_db
from pattern_recognition.data_preprocessing.speqle.concatenate_traces import CurveConcatenator

from sklearn.decomposition import PCA

In [2]:
def plot_strain_trace(trace, title):

    fig = go.Figure()

    fig.add_trace(go.Scatter(
        y=trace,
        mode='lines',
        name='Strain Trace',
        line=dict(width=3)
    ))

    fig.update_layout(
        title=title,
        xaxis_title="Timepoints",
        yaxis_title="Strain",
        template="plotly_white",
        height=500
    )

    fig.show()

In [3]:
def plot_segments_from_mean_trace(mean_trace, segment_names, title):

    n_segments = len(segment_names)
    segment_len = len(mean_trace) // n_segments

    fig = go.Figure()

    for i, name in enumerate(segment_names):
        segment = mean_trace[i * segment_len : (i + 1) * segment_len]
        fig.add_trace(go.Scatter(
            y=segment,
            mode='lines',
            name=name
        ))

    fig.update_layout(
        title=title,
        xaxis_title="Timepoints (per segment)",
        yaxis_title="Strain",
        template="plotly_white",
        height=600
    )

    fig.show()


# Phillips

In [4]:
phi_trace_loader = LoadTraces('BL', 'phi_segmentalStrainInterp_T', None, isComponents=None)
phi_DF = phi_trace_loader.tracesDF
phi_DF = phi_DF[~phi_DF['curve_segment'].str.lower().str.contains('apex', na=False)]

# GE

In [5]:
ste_trace_loader = LoadTraces('BL', 'st_segmentalStrainInterp_T', None, isComponents=None)
ste_DF = ste_trace_loader.tracesDF

## Concatenate the two DataFrames

In [6]:
contact_st_df = pd.concat([ste_DF, phi_DF], ignore_index=True)

In [7]:
contact_st_df.shape[0]

9407

In [8]:
# print(contact_st_df['strain_trace'].head(1).tolist())

In [9]:
tracesDF = contact_st_df.groupby('study_nr').filter(lambda x: len(x) == 18)
tracesDF.shape[0]

8640

In [10]:
study_counts = tracesDF.groupby('study_nr').size()
print("Number of studies traces:", len(study_counts[study_counts == 18 ]))

Number of studies traces: 480


In [11]:
tracesDF['cardiac_view'] = tracesDF['cardiac_view'].str.lower()
tracesDF['segment_wall'] = tracesDF['segment_wall'].str.lower()
dropna_tracestracesDF = tracesDF.dropna(subset=['strain_trace'])
dropna_tracestracesDF = dropna_tracestracesDF[~dropna_tracestracesDF['strain_trace'].apply(lambda x: np.all(np.array(x) == 0))]
tracestracesDF = dropna_tracestracesDF.groupby('study_nr').filter(lambda x: len(x) == 18)
tracestracesDF = tracestracesDF.sort_values(by=['study_nr', 'id_position'])
study_nr = np.sort(tracestracesDF['study_nr'].unique())

In [12]:
concatenated_strain = CurveConcatenator(tracestracesDF, isReconstructed=True)
concat_tracesDF = concatenated_strain.concatenate_traces_par()

Concatenating traces...


## Sanity check

In [13]:
# type(concat_tracesDF['concatenated_trace'].iloc[0])

In [14]:
# concat_tracesDF['trace_length'] = concat_tracesDF['concatenated_trace'].apply(len)
# length_summary = concat_tracesDF['trace_length'].value_counts().sort_index()
#
# # Print it in your desired format
# for length, count in length_summary.items():
#     print(f"{count} study_nr have {length} values in their concatenated trace")

## Compute mean across rows

In [15]:
stacked_traces = np.stack(concat_tracesDF['concatenated_trace'].values)
mean_trace = np.mean(stacked_traces, axis=0)
print(mean_trace)
len(mean_trace)

[ 0.          0.17604204  0.33685821 ... -0.60824141 -0.26036494
  0.        ]


2304

In [16]:
# plot_strain_trace(mean_trace, "Average Concatenated Strain Trace")

In [17]:
# fig.write_image("average_strain_trace.svg")

In [18]:
segment_names = concat_tracesDF['concat_curve_segm'][0].tolist()
# plot_segments_from_mean_trace(mean_trace, segment_names, "Average Concatenated Strain Trace")

In [19]:
# fig.write_image("average_segmental_strain_trace.svg")

# Load the PCA Components

In [20]:
table_name = 'calculated_pca_params_T'
recons_type = 'full_traces'
data_source = 'GE_Philips'
n_components = 59

In [21]:
pca_params_df = load_queries_table_db(dbTableNames[table_name], None, None, None)
pca_params_df = pca_params_df[pca_params_df['data_source'] == data_source]
pca_params_df = pca_params_df[pca_params_df['reconstructed_type'] == recons_type]
pca_params_df = pca_params_df[pca_params_df['n_components_'] == n_components]

In [22]:
pca_components = pca_params_df['components_']
pca_components_list = np.array(pca_components.apply(ast.literal_eval).tolist())[0]

In [23]:
# plot_strain_trace(pca_loading_vector, "loadings components_")

In [24]:
# reshaped = np.split(pca_loading_vector, 18)

In [25]:
# plot_segments_from_mean_trace(pca_loading_vector, segment_names, "Segmental component")

###  Plot the PCA components contribution

In [26]:
def get_cumulative_component_sum(pca_components_list, component_idx):
    cumulative = np.sum(pca_components_list[:component_idx + 1], axis=0)
    return cumulative


In [27]:
def generate_modified_trace(mean_trace, cumulative_component, mode='add'):
    if mode == 'add':
        return mean_trace + cumulative_component
    elif mode == 'subtract':
        return mean_trace - cumulative_component
    else:
        raise ValueError("mode must be 'add' or 'subtract'")

# Slider cumulative PCA plot

In [28]:
def plot_cumulative_pca_segmented(mean_trace, pca_components_list, segment_names, mode='add'):

    n_components = len(pca_components_list)
    n_segments = len(segment_names)
    segment_len = len(mean_trace) // n_segments

    fig = go.Figure()
    steps = []

    cumulative_component = np.zeros_like(mean_trace)

    for idx in range(n_components):
        current_component = pca_components_list[idx]
        cumulative_component += current_component

        if mode == 'add':
            modified_trace = mean_trace + cumulative_component
        elif mode == 'subtract':
            modified_trace = mean_trace - cumulative_component
        else:
            raise ValueError("mode must be 'add' or 'subtract'")


        for s_idx, seg_name in enumerate(segment_names):
            segment = modified_trace[s_idx * segment_len : (s_idx + 1) * segment_len]
            visible = (idx == 0)
            fig.add_trace(go.Scatter(
                y=segment,
                mode='lines',
                name=seg_name if idx == 0 else None,
                showlegend=(idx == 0),
                visible=visible
            ))

    for idx in range(n_components):
        visibility = [False] * (n_components * n_segments)
        start = idx * n_segments
        visibility[start:start + n_segments] = [True] * n_segments

        steps.append({
            "method": "update",
            "label": f"PCs 1–{idx+1}",
            "args": [{"visible": visibility},
                     {"title": f"Mean {('+' if mode == 'add' else '−')} PCs 1–{idx+1}"}]
        })

    fig.update_layout(
        title=f"Segment-wise Strain Curve: Cumulative PCA {'Addition' if mode == 'add' else 'Subtraction'}",
        xaxis_title="Timepoints (per segment)",
        yaxis_title="Strain",
        template="plotly_white",
        height=650,
        sliders=[{
            "active": 0,
            "currentvalue": {"prefix": "Components: "},
            "pad": {"t": 50},
            "steps": steps
        }]
    )

    fig.show()

In [29]:
plot_cumulative_pca_segmented(
    mean_trace=mean_trace,
    pca_components_list=pca_components_list,
    segment_names=concat_tracesDF['concat_curve_segm'].iloc[0],
    mode='add'  # or 'subtract'
)

# Overall mean strain contribution

In [30]:
def create_pca_video_cv2(mean_trace, pca_components_list, mode, output_path, fps):

    n_components = len(pca_components_list)
    cumulative_component = np.zeros_like(mean_trace)

    os.makedirs("frames", exist_ok=True)
    image_paths = []

    for idx in range(n_components):
        cumulative_component += pca_components_list[idx]

        if mode == 'add':
            modified_trace = mean_trace + cumulative_component
        elif mode == 'subtract':
            modified_trace = mean_trace - cumulative_component
        else:
            raise ValueError("mode must be 'add' or 'subtract'")

        fig, ax = plt.subplots(figsize=(10, 5))
        ax.plot(mean_trace, label='Mean Trace', color='gray', linestyle='--')
        ax.plot(modified_trace, label=f'{mode.title()} of PCs 1–{idx + 1}', color='blue')

        ax.set_title(f'Mean Trace {mode.title()} of PCs 1–{idx + 1}')
        ax.set_xlabel('Timepoints')
        ax.set_ylabel('Strain')
        ax.grid(True)
        ax.legend()
        fig.tight_layout()

        img_path = f"frames/frame_{idx:03d}.png"
        plt.savefig(img_path)
        plt.close()
        image_paths.append(img_path)

    # Create video
    first_img = cv2.imread(image_paths[0])
    height, width, _ = first_img.shape
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

    for img_path in image_paths:
        img = cv2.imread(img_path)
        out.write(img)

    out.release()
    print(f"✅ PCA video saved to: {output_path}")

    # Optional cleanup
    # for img_path in image_paths:
    #     os.remove(img_path)
    # os.rmdir("frames")


In [31]:
create_pca_video_cv2(
    mean_trace=mean_trace,
    pca_components_list=pca_components_list,
    mode='add',  # or 'subtract'
    output_path='pca_mean_trace_add.mp4',
    fps=2
)

✅ PCA video saved to: pca_mean_trace_add.mp4


# Video for mean segmental strain contribution

In [32]:
def create_segmental_pca_video_cv2(mean_trace, pca_components_list, segment_names, mode, output_path, fps):

    n_components = len(pca_components_list)
    n_segments = len(segment_names)
    segment_len = len(mean_trace) // n_segments
    cumulative_component = np.zeros_like(mean_trace)

    os.makedirs("segment_frames", exist_ok=True)
    image_paths = []

    for idx in range(n_components):
        cumulative_component += pca_components_list[idx]
        if mode == 'add':
            modified_trace = mean_trace + cumulative_component
        elif mode == 'subtract':
            modified_trace = mean_trace - cumulative_component
        else:
            raise ValueError("mode must be 'add' or 'subtract'")

        fig, ax = plt.subplots(figsize=(10, 5))
        for s_idx, name in enumerate(segment_names):
            segment = modified_trace[s_idx * segment_len : (s_idx + 1) * segment_len]
            ax.plot(segment, label=name)

        ax.set_title(f'Segmental Mean {mode.title()} of PCs 1–{idx + 1}')
        ax.set_xlabel('Timepoints')
        ax.set_ylabel('Strain')
        ax.grid(True)
        ax.legend(fontsize=7, ncol=3, loc='lower left')
        fig.tight_layout()

        img_path = f"segment_frames/frame_{idx:03d}.png"
        plt.savefig(img_path)
        plt.close()
        image_paths.append(img_path)

    # Create video with cv2
    first_img = cv2.imread(image_paths[0])
    height, width, _ = first_img.shape
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

    for img_path in image_paths:
        img = cv2.imread(img_path)
        out.write(img)

    out.release()
    print(f"✅ Segmental PCA video saved to: {output_path}")

    # Optional cleanup
    # for img_path in image_paths:
    #     os.remove(img_path)
    # os.rmdir("segment_frames")


In [33]:
create_segmental_pca_video_cv2(
    mean_trace=mean_trace,
    pca_components_list=pca_components_list,
    segment_names=concat_tracesDF['concat_curve_segm'].iloc[0],
    mode='subtract',        # or 'subtract' 'add'
    output_path='segmental_pca_subtract.mp4',
    fps=3              # slower video
)


✅ Segmental PCA video saved to: segmental_pca_subtract.mp4


In [34]:
create_segmental_pca_video_cv2(
    mean_trace=mean_trace,
    pca_components_list=pca_components_list,
    segment_names=concat_tracesDF['concat_curve_segm'].iloc[0],
    mode='add',        # or 'subtract' 'add'
    output_path='segmental_pca_add.mp4',
    fps=3              # slower video
)


✅ Segmental PCA video saved to: segmental_pca_add.mp4


# Computing the PCA components contribution

In [35]:
n_components = 59  # or whatever number you used before
pca = PCA(n_components=n_components)
X_transformed = pca.fit_transform(stacked_traces)  # shape: (n_patients, n_components)


In [36]:
components_ = pca.components_            # shape: (n_components, n_timepoints)
explained_variance = pca.explained_variance_ratio_  # shape: (n_components,)
mean_vector = pca.mean_                  # shape: (n_timepoints,)

In [37]:
create_pca_video_cv2(
    mean_trace=mean_trace,
    pca_components_list=components_,
    mode='add',  # or 'subtract'
    output_path='pca_mean_trace_add_pca.mp4',
    fps=2
)

✅ PCA video saved to: pca_mean_trace_add_pca.mp4


In [38]:
# cos_sim = lambda a, b: np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))
#
# for i in range(5):
#     sim = cos_sim(components_[i], pca_components_list[i])
#     print(f"PC{i+1} cosine similarity: {sim:.4f}")

In [39]:


def plot_cumulative_pca_reconstruction_plotly(pca, mean_trace, segment_names=None):

    # Step 1: Center mean_trace and project into PCA space
    mean_centered = mean_trace - pca.mean_
    mean_projection = mean_centered @ pca.components_.T  # shape: (n_components,)
    components = pca.components_  # shape: (n_components, n_timepoints)

    # Step 2: Reconstruct cumulative components
    cumulative_reconstructions = []
    cumulative_sum = np.zeros_like(mean_centered)

    for i in range(pca.n_components_):
        cumulative_sum += mean_projection[i] * components[i]
        reconstructed_trace = cumulative_sum + pca.mean_
        cumulative_reconstructions.append(reconstructed_trace)

    # Step 3: Plot cumulative reconstructions
    fig = go.Figure()

    for i, curve in enumerate(cumulative_reconstructions):
        fig.add_trace(go.Scatter(
            y=curve,
            mode='lines',
            name=f'Cumulative PC1–{i+1}',
            visible=(i == 0),  # show only the first by default
            line=dict(width=2)
        ))

    # Add true mean trace for comparison
    fig.add_trace(go.Scatter(
        y=mean_trace,
        mode='lines',
        name='True Mean Trace',
        line=dict(width=3, color='black')
    ))

    # Optional segment lines
    if segment_names is not None:
        n_segments = len(segment_names)
        segment_len = len(mean_trace) // n_segments
        for i in range(1, n_segments):
            fig.add_vline(
                x=i * segment_len,
                line=dict(color="gray", width=1, dash="dot")
            )

    # Slider for visualizing additions
    steps = []
    for i in range(pca.n_components_):
        visibility = [False] * (pca.n_components_ + 1)  # +1 for true mean
        visibility[i] = True
        visibility[-1] = True  # always show mean

        steps.append(dict(
            method="update",
            label=f"PCs 1–{i+1}",
            args=[{"visible": visibility},
                  {"title": f"Cumulative PCA Reconstruction: PCs 1–{i+1}"}]
        ))

    sliders = [dict(
        active=0,
        currentvalue={"prefix": "Showing: "},
        pad={"t": 50},
        steps=steps
    )]

    fig.update_layout(
        title="Cumulative PCA Reconstruction of Mean Strain Trace",
        xaxis_title="Timepoints",
        yaxis_title="Strain",
        template="plotly_white",
        sliders=sliders,
        height=700
    )

    fig.show()


In [40]:
plot_cumulative_pca_reconstruction_plotly(
    pca=pca,
    mean_trace=mean_trace,
    segment_names=concat_tracesDF['concat_curve_segm'].iloc[0]
)


In [41]:
import numpy as np
import plotly.graph_objects as go

def plot_pca_reconstruction_error_plotly(pca, mean_trace, max_components=10):
    """
    Visualize how the cumulative PCA reconstruction approximates the mean trace, and how the difference decreases.

    Parameters:
    - pca: fitted sklearn PCA object
    - mean_trace: 1D array, original mean strain trace
    - max_components: int, number of steps to show (e.g., 10)
    """
    mean_centered = mean_trace - pca.mean_
    mean_projection = mean_centered @ pca.components_.T
    components = pca.components_

    cumulative_sum = np.zeros_like(mean_centered)
    reconstructions = []
    differences = []

    for i in range(max_components):
        cumulative_sum += mean_projection[i] * components[i]
        reconstruction = cumulative_sum + pca.mean_
        reconstructions.append(reconstruction)
        differences.append(mean_trace - reconstruction)

    fig = go.Figure()

    for i in range(max_components):
        fig.add_trace(go.Scatter(
            y=reconstructions[i],
            mode='lines',
            name=f'Reconstruction (PCs 1–{i+1})',
            visible=(i == 0),
            line=dict(width=2)
        ))

    # Always plot original mean trace
    fig.add_trace(go.Scatter(
        y=mean_trace,
        mode='lines',
        name='Original Mean Trace',
        line=dict(width=3, color='black')
    ))

    # Slider setup
    steps = []
    for i in range(max_components):
        visible = [False] * (max_components + 1)
        visible[i] = True  # Show only this reconstruction
        visible[-1] = True  # Keep original mean always visible

        steps.append(dict(
            method='update',
            label=f'PCs 1–{i+1}',
            args=[{"visible": visible},
                  {"title": f'Reconstruction using PCs 1–{i+1}'}]
        ))

    fig.update_layout(
        title="Progressive PCA Reconstruction of Mean Strain",
        xaxis_title="Timepoints",
        yaxis_title="Strain",
        sliders=[dict(
            active=0,
            currentvalue={"prefix": "Using: "},
            pad={"t": 50},
            steps=steps
        )],
        template='plotly_white',
        height=600
    )

    fig.show()

    # Optional: return reconstruction error for plotting
    return differences


In [42]:
plot_pca_reconstruction_error_plotly(
    pca=pca,
    mean_trace=mean_trace,
    max_components=10  # Try 10 to 20 first for visibility
)


[array([0., 0., 0., ..., 0., 0., 0.]),
 array([0., 0., 0., ..., 0., 0., 0.]),
 array([0., 0., 0., ..., 0., 0., 0.]),
 array([0., 0., 0., ..., 0., 0., 0.]),
 array([0., 0., 0., ..., 0., 0., 0.]),
 array([0., 0., 0., ..., 0., 0., 0.]),
 array([0., 0., 0., ..., 0., 0., 0.]),
 array([0., 0., 0., ..., 0., 0., 0.]),
 array([0., 0., 0., ..., 0., 0., 0.]),
 array([0., 0., 0., ..., 0., 0., 0.])]